In [ ]:
#| default_exp cli.ingest

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from fastcore.script import *
from typing import Optional, Literal
from fastcore.style import S
import sys
import importlib
from importlib import import_module
from pkgutil import iter_modules

In [ ]:
#| export
def import_handler(handler_name:str,  # Name of the handler module
                   fn_name:str='encode'  # Name of the function to import
                  )->callable:  # The requested function
    "Import `fn_name` from module `handler_name`"
    try:
        handler = import_module(handler_name)
        return getattr(handler, fn_name)
    except (ImportError, AttributeError): 
        print(f"Failed to import function: {fn_name}")

In [ ]:
#| export
def handlers() -> dict:
    "Dict of available handlers and their status"
    return {m.name:getattr(import_module(f'marisco.handlers.{m.name}'), 'status', 'Active') for m in iter_modules(marisco.handlers.__path__)}

In [ ]:
#| eval: false
handlers()

{'fram_strait2025': 'Active',
 'geotraces': 'Active',
 'helcom': 'Active',
 'jois': 'Active',
 'maris_legacy': 'Active',
 'ospar': 'Under refactoring',
 'tepco': 'Under refactoring'}

In [ ]:
#| export
@call_parse
def main(
    ds: str,  # Name of the dataset to encode as NetCDF4; see `handlers()` for available names
    dest: str, # Output path: file path for handlers that write one file, or folder path for handlers that write several (e.g. maris_legacy)
    src: Optional[str] = None,  # Optional path to local input data; only needed by handlers that don't fetch data online
    **kwargs,  # Additional arguments passed through to the handler's `encode` (e.g. ref_ids for maris_legacy)
) -> None:
    "Convert a marine radioactivity dataset to MARIS NetCDF4 format."
    hs = handlers()
    if ds not in hs:
        print(S.red(f"Invalid handler name: {ds}. Available handlers: {', '.join(hs)}"))
        sys.exit(1)
    if hs[ds] != 'Active':
        print(S.yellow(f"Warning: {ds} is {hs[ds]}"))
        sys.exit(1)
    encode = import_handler(f'marisco.handlers.{ds}')
    encode(dest=dest, src=src, **kwargs)


For instance, for a single-file output (helcom, geotraces, tepco, ...):

`maris_to_nc helcom output/100-HELCOM-MORS-2024.nc`

or folder output with a ref_ids subset (maris_legacy):

`maris_to_nc maris_legacy ~/output ~/data/maris/dump.txt --ref_ids "16,30"`
